# Lab 10 - K-Means Clustering
## Heart Disease Data Clustering using K-Means Algorithm

**Objective:** To implement K-Means clustering on the Cleveland Heart Disease dataset and analyze the natural groupings in the data.

### 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
import warnings
warnings.filterwarnings('ignore')

### 2. Load the Dataset

In [ ]:
# Define column names for the Cleveland Heart Disease dataset
columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
           'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

# Load the dataset
df = pd.read_csv('processed.cleveland.data', header=None, names=columns, na_values='?')

print("Shape of dataset:", df.shape)
df.head()

### 3. Data Preprocessing

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

# Handle missing values
df['ca'] = pd.to_numeric(df['ca'], errors='coerce')
df['thal'] = pd.to_numeric(df['thal'], errors='coerce')
df.fillna(df.median(), inplace=True)

# Convert target to binary
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

print("\nMissing values after handling:")
print(df.isnull().sum().sum())

In [ ]:
# Separate features (we'll use features for clustering, target for comparison)
X = df.drop('target', axis=1)
y_actual = df['target']

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled features shape:", X_scaled.shape)
print("\nScaled data sample:")
print(pd.DataFrame(X_scaled, columns=X.columns).head())

### 4. Finding Optimal K using Elbow Method

In [ ]:
# Elbow Method
inertias = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Inertia (Within-cluster sum of squares)', fontsize=12)
plt.title('Elbow Method for Optimal K', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(K_range)
plt.show()

### 5. Silhouette Score Analysis

In [ ]:
# Silhouette Score for different K values
silhouette_scores = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f"K={k}: Silhouette Score = {score:.4f}")

plt.figure(figsize=(10, 6))
plt.plot(K_range, silhouette_scores, 'rs-', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.title('Silhouette Score for Different K Values', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(K_range)
plt.show()

best_k = K_range[np.argmax(silhouette_scores)]
print(f"\nBest K based on Silhouette Score: {best_k}")

### 6. Apply K-Means with K=2 (Disease vs No Disease)

In [ ]:
# Apply K-Means with K=2
kmeans_2 = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels = kmeans_2.fit_predict(X_scaled)

print("Cluster distribution:")
print(pd.Series(cluster_labels).value_counts())

# Compare with actual labels
ari_score = adjusted_rand_score(y_actual, cluster_labels)
print(f"\nAdjusted Rand Index (comparison with actual labels): {ari_score:.4f}")

### 7. PCA Visualization of Clusters

In [ ]:
# Reduce dimensions using PCA for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_):.4f}")

# Plot clusters vs actual labels side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# K-Means Clusters
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='viridis', alpha=0.6, s=50)
centers = pca.transform(kmeans_2.cluster_centers_)
axes[0].scatter(centers[:, 0], centers[:, 1], c='red', marker='X', s=200, edgecolors='black', linewidth=2)
axes[0].set_title('K-Means Clusters (K=2)', fontsize=14)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
plt.colorbar(scatter1, ax=axes[0])

# Actual Labels
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_actual, cmap='coolwarm', alpha=0.6, s=50)
axes[1].set_title('Actual Labels (Disease vs No Disease)', fontsize=14)
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
plt.colorbar(scatter2, ax=axes[1])

plt.tight_layout()
plt.show()

### 8. Cluster Analysis

In [ ]:
# Add cluster labels to the dataframe
df['cluster'] = cluster_labels

# Analyze cluster characteristics
print("Cluster 0 Statistics:")
print(df[df['cluster'] == 0].describe().round(2))
print("\n" + "="*80)
print("\nCluster 1 Statistics:")
print(df[df['cluster'] == 1].describe().round(2))

In [ ]:
# Compare cluster means for key features
cluster_means = df.groupby('cluster')[['age', 'trestbps', 'chol', 'thalach', 'oldpeak']].mean()

cluster_means.plot(kind='bar', figsize=(12, 6), colormap='Set2')
plt.title('Cluster Comparison - Mean Feature Values', fontsize=14)
plt.xlabel('Cluster')
plt.ylabel('Mean Value')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Cross-tabulation of clusters vs actual target
cross_tab = pd.crosstab(df['cluster'], df['target'], margins=True)
print("Cross-tabulation: Clusters vs Actual Labels")
print(cross_tab)

### 9. Conclusion

In this lab, we applied K-Means clustering to the Cleveland Heart Disease dataset. We used the Elbow Method and Silhouette Score to determine the optimal number of clusters. With K=2, we compared the clusters with actual disease labels using PCA visualization and the Adjusted Rand Index. The analysis shows how unsupervised learning can discover natural groupings in clinical data that partially align with disease status.